In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_path = "CMAPSSData.zip"
extract_path = "CMAPSSData"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:")
print(os.listdir(extract_path))

In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from google.colab import files

In [ ]:
columns = (
    ["unit_number", "time_cycles"] +
    [f"setting_{i}" for i in range(1, 4)] +
    [f"sensor_{i}" for i in range(1, 22)]
)

train_path = f"{extract_path}/train_FD001.txt"
test_path = f"{extract_path}/test_FD001.txt"
rul_path = f"{extract_path}/RUL_FD001.txt"

train_df_original = pd.read_csv(
    train_path,
    sep=r"\s+",
    header=None,
    names=columns
)

test_df_original = pd.read_csv(
    test_path,
    sep=r"\s+",
    header=None,
    names=columns
)

true_rul_df = pd.read_csv(
    rul_path,
    sep=r"\s+",
    header=None,
    names=["true_RUL"]
)

print("Training data shape:", train_df_original.shape)
print("Testing data shape:", test_df_original.shape)
print("Actual NASA RUL shape:", true_rul_df.shape)

train_df_original.head()

In [ ]:
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
setting_cols = [f"setting_{i}" for i in range(1, 4)]
base_feature_cols = ["time_cycles"] + setting_cols + sensor_cols

In [ ]:
def create_rul_labels(train_df, rul_cap):
    train_df = train_df.copy()

    max_cycles = train_df.groupby("unit_number")["time_cycles"].max()

    train_df["RUL"] = train_df.apply(
        lambda row: max_cycles[row["unit_number"]] - row["time_cycles"],
        axis=1
    )

    train_df["RUL_capped"] = train_df["RUL"].clip(upper=rul_cap)

    return train_df

In [ ]:
def calculate_slope(values):
    if len(values) < 2:
        return 0

    x = np.arange(len(values))
    y = np.array(values)

    slope = np.polyfit(x, y, 1)[0]
    return slope


def add_engineered_features(df):
    df = df.copy()
    df = df.sort_values(["unit_number", "time_cycles"])

    for sensor in sensor_cols:
        # Rolling mean over last 5 cycles
        df[f"{sensor}_rolling_mean_5"] = (
            df.groupby("unit_number")[sensor]
            .rolling(window=5, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        # Rolling standard deviation over last 5 cycles
        df[f"{sensor}_rolling_std_5"] = (
            df.groupby("unit_number")[sensor]
            .rolling(window=5, min_periods=1)
            .std()
            .reset_index(level=0, drop=True)
        )

        # Difference from previous cycle
        df[f"{sensor}_diff"] = (
            df.groupby("unit_number")[sensor]
            .diff()
        )

        # Slope over last 10 cycles
        df[f"{sensor}_slope_10"] = (
            df.groupby("unit_number")[sensor]
            .rolling(window=10, min_periods=2)
            .apply(calculate_slope, raw=False)
            .reset_index(level=0, drop=True)
        )

        # Deviation from early healthy baseline for each engine
        df[f"{sensor}_baseline_deviation"] = (
            df[sensor] -
            df.groupby("unit_number")[sensor].transform(
                lambda x: x.iloc[:min(20, len(x))].mean()
            )
        )

    df = df.fillna(0)

    return df

In [ ]:
engineered_feature_cols = []

for sensor in sensor_cols:
    engineered_feature_cols.extend([
        f"{sensor}_rolling_mean_5",
        f"{sensor}_rolling_std_5",
        f"{sensor}_diff",
        f"{sensor}_slope_10",
        f"{sensor}_baseline_deviation"
    ])

tree_feature_cols = base_feature_cols + engineered_feature_cols

print("Total tree model features:", len(tree_feature_cols))

In [ ]:
def get_tree_models():
    models = {
        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),

        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),

        "Extra Trees": ExtraTreesRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": xgb.XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    return models

In [ ]:
def get_risk_category(rul):
    if rul <= 30:
        return "High Risk"
    elif rul <= 60:
        return "Moderate Risk"
    else:
        return "Low Risk"

In [ ]:
rul_caps_to_test = [100, 125, 150]

all_tree_results = []
all_tree_prediction_tables = {}
all_trained_tree_models = {}

for rul_cap in rul_caps_to_test:
    print(f"\n==============================")
    print(f"Testing RUL_CAP = {rul_cap}")
    print(f"==============================")

    train_df = create_rul_labels(train_df_original, rul_cap)
    test_df = test_df_original.copy()

    train_features_df = add_engineered_features(train_df)
    test_features_df = add_engineered_features(test_df)

    X = train_features_df[tree_feature_cols]
    y = train_features_df["RUL_capped"]
    groups = train_features_df["unit_number"]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, val_idx = next(splitter.split(X, y, groups=groups))

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    last_test_cycles = test_features_df.groupby("unit_number").tail(1).copy()
    X_test_last = last_test_cycles[tree_feature_cols]
    actual_rul = true_rul_df["true_RUL"].clip(upper=rul_cap)

    prediction_table = pd.DataFrame({
        "engine_id": last_test_cycles["unit_number"].values,
        "last_cycle": last_test_cycles["time_cycles"].values,
        "actual_RUL": actual_rul.values
    })

    models = get_tree_models()
    trained_models = {}

    for model_name, model in models.items():
        print(f"Training {model_name} with RUL_CAP={rul_cap}...")
        model.fit(X_train, y_train)
        trained_models[model_name] = model

        val_preds = model.predict(X_val)
        test_preds = model.predict(X_test_last)
        test_preds = np.clip(test_preds, 0, rul_cap)

        val_mae = mean_absolute_error(y_val, val_preds)
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        val_r2 = r2_score(y_val, val_preds)

        test_mae = mean_absolute_error(actual_rul, test_preds)
        test_rmse = np.sqrt(mean_squared_error(actual_rul, test_preds))
        test_r2 = r2_score(actual_rul, test_preds)

        prediction_table[f"{model_name}_predicted_RUL"] = test_preds
        prediction_table[f"{model_name}_absolute_error"] = abs(test_preds - actual_rul.values)
        prediction_table[f"{model_name}_predicted_risk"] = prediction_table[f"{model_name}_predicted_RUL"].apply(get_risk_category)

        prediction_table["actual_risk"] = prediction_table["actual_RUL"].apply(get_risk_category)

        risk_accuracy = (
            prediction_table[f"{model_name}_predicted_risk"] == prediction_table["actual_risk"]
        ).mean() * 100

        all_tree_results.append({
            "RUL_CAP": rul_cap,
            "model": model_name,
            "validation_MAE": val_mae,
            "validation_RMSE": val_rmse,
            "validation_R2": val_r2,
            "test_MAE": test_mae,
            "test_RMSE": test_rmse,
            "test_R2": test_r2,
            "risk_category_accuracy_percent": risk_accuracy
        })

    all_tree_prediction_tables[rul_cap] = prediction_table
    all_trained_tree_models[rul_cap] = trained_models

tree_results_df = pd.DataFrame(all_tree_results)
tree_results_df = tree_results_df.sort_values("test_MAE")

tree_results_df

In [ ]:
best_tree_row = tree_results_df.iloc[0]

best_tree_cap = int(best_tree_row["RUL_CAP"])
best_tree_model_name = best_tree_row["model"]

print("Best tree model:", best_tree_model_name)
print("Best RUL cap:", best_tree_cap)
print("Best tree test MAE:", best_tree_row["test_MAE"])

best_tree_model = all_trained_tree_models[best_tree_cap][best_tree_model_name]
best_tree_prediction_df = all_tree_prediction_tables[best_tree_cap].copy()

best_tree_prediction_df.head()

In [ ]:
tree_results_df.to_csv("tree_model_results_all_caps.csv", index=False)
best_tree_prediction_df.to_csv("best_tree_model_predictions.csv", index=False)

files.download("tree_model_results_all_caps.csv")
files.download("best_tree_model_predictions.csv")

In [ ]:
RUL_CAP = best_tree_cap

train_df = create_rul_labels(train_df_original, RUL_CAP)
test_df = test_df_original.copy()

sequence_feature_cols = setting_cols + sensor_cols

print("Using RUL_CAP for LSTM:", RUL_CAP)
print("Number of sequence features:", len(sequence_feature_cols))

In [ ]:
unique_engines = train_df["unit_number"].unique()

np.random.seed(42)
np.random.shuffle(unique_engines)

split_point = int(0.8 * len(unique_engines))

train_engines = unique_engines[:split_point]
val_engines = unique_engines[split_point:]

lstm_train_df = train_df[train_df["unit_number"].isin(train_engines)].copy()
lstm_val_df = train_df[train_df["unit_number"].isin(val_engines)].copy()

print("LSTM train engines:", len(train_engines))
print("LSTM validation engines:", len(val_engines))

In [ ]:
scaler = StandardScaler()

scaler.fit(lstm_train_df[sequence_feature_cols])

lstm_train_df[sequence_feature_cols] = scaler.transform(
    lstm_train_df[sequence_feature_cols]
)

lstm_val_df[sequence_feature_cols] = scaler.transform(
    lstm_val_df[sequence_feature_cols]
)

test_scaled_df = test_df.copy()
test_scaled_df[sequence_feature_cols] = scaler.transform(
    test_scaled_df[sequence_feature_cols]
)

In [ ]:
WINDOW_SIZE = 30

def create_lstm_sequences(df, feature_cols, target_col, window_size=30):
    X_sequences = []
    y_sequences = []

    for engine_id in df["unit_number"].unique():
        engine_data = df[df["unit_number"] == engine_id].sort_values("time_cycles")

        feature_values = engine_data[feature_cols].values
        target_values = engine_data[target_col].values

        for i in range(window_size, len(engine_data) + 1):
            X_sequences.append(feature_values[i-window_size:i])
            y_sequences.append(target_values[i-1])

    return np.array(X_sequences), np.array(y_sequences)


X_lstm_train, y_lstm_train = create_lstm_sequences(
    lstm_train_df,
    sequence_feature_cols,
    "RUL_capped",
    WINDOW_SIZE
)

X_lstm_val, y_lstm_val = create_lstm_sequences(
    lstm_val_df,
    sequence_feature_cols,
    "RUL_capped",
    WINDOW_SIZE
)

print("LSTM training shape:", X_lstm_train.shape)
print("LSTM validation shape:", X_lstm_val.shape)

In [ ]:
tf.random.set_seed(42)

lstm_model = Sequential([
    LSTM(
        64,
        input_shape=(WINDOW_SIZE, len(sequence_feature_cols)),
        return_sequences=False
    ),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=75,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(history.history["mae"], label="Training MAE")
plt.plot(history.history["val_mae"], label="Validation MAE")

plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("LSTM Training vs Validation MAE")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("lstm_training_history.png", dpi=300)
files.download("lstm_training_history.png")

plt.show()

In [ ]:
def create_lstm_test_sequences(df, feature_cols, window_size=30):
    X_test_sequences = []
    engine_ids = []

    for engine_id in df["unit_number"].unique():
        engine_data = df[df["unit_number"] == engine_id].sort_values("time_cycles")
        feature_values = engine_data[feature_cols].values

        if len(engine_data) >= window_size:
            final_window = feature_values[-window_size:]
        else:
            pad_size = window_size - len(engine_data)
            padding = np.repeat(feature_values[0:1], pad_size, axis=0)
            final_window = np.vstack([padding, feature_values])

        X_test_sequences.append(final_window)
        engine_ids.append(engine_id)

    return np.array(X_test_sequences), np.array(engine_ids)


X_lstm_test, lstm_test_engine_ids = create_lstm_test_sequences(
    test_scaled_df,
    sequence_feature_cols,
    WINDOW_SIZE
)

print("LSTM test shape:", X_lstm_test.shape)

In [ ]:
lstm_test_preds = lstm_model.predict(X_lstm_test).flatten()
lstm_test_preds = np.clip(lstm_test_preds, 0, RUL_CAP)

actual_rul = true_rul_df["true_RUL"].clip(upper=RUL_CAP)

lstm_mae = mean_absolute_error(actual_rul, lstm_test_preds)
lstm_rmse = np.sqrt(mean_squared_error(actual_rul, lstm_test_preds))
lstm_r2 = r2_score(actual_rul, lstm_test_preds)

lstm_predicted_risk = pd.Series(lstm_test_preds).apply(get_risk_category)
actual_risk = actual_rul.apply(get_risk_category)

lstm_risk_accuracy = (lstm_predicted_risk.values == actual_risk.values).mean() * 100

lstm_results = pd.DataFrame({
    "RUL_CAP": [RUL_CAP],
    "model": ["LSTM"],
    "test_MAE": [lstm_mae],
    "test_RMSE": [lstm_rmse],
    "test_R2": [lstm_r2],
    "risk_category_accuracy_percent": [lstm_risk_accuracy]
})

lstm_results

In [ ]:
final_prediction_df = best_tree_prediction_df.copy()

final_prediction_df["LSTM_predicted_RUL"] = lstm_test_preds
final_prediction_df["LSTM_absolute_error"] = abs(
    final_prediction_df["LSTM_predicted_RUL"] - final_prediction_df["actual_RUL"]
)
final_prediction_df["LSTM_predicted_risk"] = final_prediction_df["LSTM_predicted_RUL"].apply(get_risk_category)

final_prediction_df.head()

In [ ]:
tree_pred_col = f"{best_tree_model_name}_predicted_RUL"

final_prediction_df["Ensemble_predicted_RUL"] = (
    0.5 * final_prediction_df[tree_pred_col] +
    0.5 * final_prediction_df["LSTM_predicted_RUL"]
)

final_prediction_df["Ensemble_predicted_RUL"] = final_prediction_df["Ensemble_predicted_RUL"].clip(0, RUL_CAP)

final_prediction_df["Ensemble_absolute_error"] = abs(
    final_prediction_df["Ensemble_predicted_RUL"] - final_prediction_df["actual_RUL"]
)

final_prediction_df["Ensemble_predicted_risk"] = final_prediction_df["Ensemble_predicted_RUL"].apply(get_risk_category)

ensemble_mae = mean_absolute_error(
    final_prediction_df["actual_RUL"],
    final_prediction_df["Ensemble_predicted_RUL"]
)

ensemble_rmse = np.sqrt(mean_squared_error(
    final_prediction_df["actual_RUL"],
    final_prediction_df["Ensemble_predicted_RUL"]
))

ensemble_r2 = r2_score(
    final_prediction_df["actual_RUL"],
    final_prediction_df["Ensemble_predicted_RUL"]
)

ensemble_risk_accuracy = (
    final_prediction_df["Ensemble_predicted_risk"] == final_prediction_df["actual_risk"]
).mean() * 100

ensemble_results = pd.DataFrame({
    "RUL_CAP": [RUL_CAP],
    "model": [f"Ensemble: {best_tree_model_name} + LSTM"],
    "test_MAE": [ensemble_mae],
    "test_RMSE": [ensemble_rmse],
    "test_R2": [ensemble_r2],
    "risk_category_accuracy_percent": [ensemble_risk_accuracy]
})

ensemble_results

In [ ]:
best_tree_summary = tree_results_df[
    (tree_results_df["RUL_CAP"] == best_tree_cap) &
    (tree_results_df["model"] == best_tree_model_name)
][
    [
        "RUL_CAP",
        "model",
        "test_MAE",
        "test_RMSE",
        "test_R2",
        "risk_category_accuracy_percent"
    ]
]

final_model_comparison = pd.concat(
    [
        best_tree_summary,
        lstm_results,
        ensemble_results
    ],
    ignore_index=True
)

final_model_comparison = final_model_comparison.sort_values("test_MAE")

final_model_comparison

In [ ]:
final_model_comparison.to_csv("final_advanced_model_comparison.csv", index=False)
final_prediction_df.to_csv("final_actual_vs_predicted_advanced_models.csv", index=False)

files.download("final_advanced_model_comparison.csv")
files.download("final_actual_vs_predicted_advanced_models.csv")

In [ ]:
sorted_df = final_prediction_df.sort_values("actual_RUL").reset_index(drop=True)

plt.figure(figsize=(18, 8))

plt.plot(
    sorted_df.index,
    sorted_df["actual_RUL"],
    marker="o",
    linewidth=2,
    label="Actual NASA RUL"
)

plt.plot(
    sorted_df.index,
    sorted_df[f"{best_tree_model_name}_predicted_RUL"],
    linestyle="--",
    alpha=0.8,
    label=f"Best Tree Model: {best_tree_model_name}"
)

plt.plot(
    sorted_df.index,
    sorted_df["LSTM_predicted_RUL"],
    linestyle="--",
    alpha=0.8,
    label="LSTM Prediction"
)

plt.plot(
    sorted_df.index,
    sorted_df["Ensemble_predicted_RUL"],
    linestyle="-.",
    linewidth=2,
    alpha=0.9,
    label="Ensemble Prediction"
)

plt.xlabel("Test Engines Sorted by Actual RUL")
plt.ylabel("Remaining Useful Life, RUL")
plt.title("Advanced Model Comparison: Tree Model vs LSTM vs Ensemble")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("advanced_model_comparison_sorted.png", dpi=300)
files.download("advanced_model_comparison_sorted.png")

plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(
    final_prediction_df["engine_id"],
    final_prediction_df[f"{best_tree_model_name}_absolute_error"],
    marker="o",
    alpha=0.8,
    label=f"{best_tree_model_name} Absolute Error"
)

plt.plot(
    final_prediction_df["engine_id"],
    final_prediction_df["LSTM_absolute_error"],
    marker="s",
    alpha=0.8,
    label="LSTM Absolute Error"
)

plt.plot(
    final_prediction_df["engine_id"],
    final_prediction_df["Ensemble_absolute_error"],
    marker="x",
    alpha=0.8,
    label="Ensemble Absolute Error"
)

plt.xlabel("Engine ID")
plt.ylabel("Absolute Error in RUL Cycles")
plt.title("Advanced Model Error Comparison by Engine")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("advanced_model_error_comparison.png", dpi=300)
files.download("advanced_model_error_comparison.png")

plt.show()

In [ ]:
final_prediction_df["LSTM_improvement_over_tree"] = (
    final_prediction_df[f"{best_tree_model_name}_absolute_error"] -
    final_prediction_df["LSTM_absolute_error"]
)

lstm_improvement_cases = final_prediction_df.sort_values(
    "LSTM_improvement_over_tree",
    ascending=False
)

lstm_improvement_cases[
    [
        "engine_id",
        "actual_RUL",
        f"{best_tree_model_name}_predicted_RUL",
        "LSTM_predicted_RUL",
        f"{best_tree_model_name}_absolute_error",
        "LSTM_absolute_error",
        "LSTM_improvement_over_tree"
    ]
].head(10)

In [ ]:
final_prediction_df["ensemble_improvement_over_tree"] = (
    final_prediction_df[f"{best_tree_model_name}_absolute_error"] -
    final_prediction_df["Ensemble_absolute_error"]
)

ensemble_improvement_cases = final_prediction_df.sort_values(
    "ensemble_improvement_over_tree",
    ascending=False
)

ensemble_improvement_cases[
    [
        "engine_id",
        "actual_RUL",
        f"{best_tree_model_name}_predicted_RUL",
        "LSTM_predicted_RUL",
        "Ensemble_predicted_RUL",
        f"{best_tree_model_name}_absolute_error",
        "LSTM_absolute_error",
        "Ensemble_absolute_error",
        "ensemble_improvement_over_tree"
    ]
].head(10)

In [ ]:
final_prediction_df["average_advanced_error"] = final_prediction_df[
    [
        f"{best_tree_model_name}_absolute_error",
        "LSTM_absolute_error",
        "Ensemble_absolute_error"
    ]
].mean(axis=1)

hard_cases = final_prediction_df.sort_values(
    "average_advanced_error",
    ascending=False
)

hard_cases[
    [
        "engine_id",
        "actual_RUL",
        f"{best_tree_model_name}_predicted_RUL",
        "LSTM_predicted_RUL",
        "Ensemble_predicted_RUL",
        f"{best_tree_model_name}_absolute_error",
        "LSTM_absolute_error",
        "Ensemble_absolute_error",
        "average_advanced_error"
    ]
].head(10)

In [ ]:
def review_engine_advanced(engine_id):
    row = final_prediction_df[final_prediction_df["engine_id"] == engine_id].iloc[0]

    print("Advanced Engine Review")
    print("======================")
    print(f"Engine ID: {engine_id}")
    print(f"Actual NASA RUL: {row['actual_RUL']:.1f} cycles")
    print(f"Actual Risk Category: {row['actual_risk']}")
    print()

    model_names = [
        best_tree_model_name,
        "LSTM",
        "Ensemble"
    ]

    for name in model_names:
        print(f"{name}:")
        print(f"  Predicted RUL: {row[f'{name}_predicted_RUL']:.1f} cycles")
        print(f"  Absolute Error: {row[f'{name}_absolute_error']:.1f} cycles")
        print(f"  Predicted Risk: {row[f'{name}_predicted_risk']}")
        print()

    errors = {
        name: row[f"{name}_absolute_error"]
        for name in model_names
    }

    best_for_engine = min(errors, key=errors.get)

    print(f"Best model for this specific engine: {best_for_engine}")

In [ ]:
review_engine_advanced(1)

In [ ]:
review_engine_advanced(25)
review_engine_advanced(80)
review_engine_advanced(93)

In [ ]:
def plot_engine_sensor_trends(engine_id, sensors_to_plot=None):
    if sensors_to_plot is None:
        sensors_to_plot = [
            "sensor_2",
            "sensor_3",
            "sensor_4",
            "sensor_7",
            "sensor_11",
            "sensor_12",
            "sensor_15"
        ]

    engine_data = test_df_original[test_df_original["unit_number"] == engine_id]

    if engine_data.empty:
        print(f"Engine {engine_id} not found.")
        return

    plt.figure(figsize=(14, 7))

    for sensor in sensors_to_plot:
        plt.plot(
            engine_data["time_cycles"],
            engine_data[sensor],
            label=sensor
        )

    plt.xlabel("Time Cycle")
    plt.ylabel("Sensor Value")
    plt.title(f"Sensor Trends Over Time for Engine {engine_id}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    filename = f"engine_{engine_id}_sensor_trends.png"
    plt.savefig(filename, dpi=300)
    files.download(filename)

    plt.show()

In [ ]:
plot_engine_sensor_trends(1)

In [ ]:
plot_engine_sensor_trends(25)
plot_engine_sensor_trends(80)
plot_engine_sensor_trends(93)

In [ ]:
def maintenance_report_advanced(engine_id):
    row = final_prediction_df[final_prediction_df["engine_id"] == engine_id]

    if row.empty:
        return f"Engine {engine_id} was not found."

    row = row.iloc[0]

    predicted_rul = row["Ensemble_predicted_RUL"]
    risk = row["Ensemble_predicted_risk"]
    current_cycle = row["last_cycle"]

    if risk == "High Risk":
        recommendation = "Schedule maintenance or detailed inspection as soon as possible."
    elif risk == "Moderate Risk":
        recommendation = "Increase monitoring frequency and prepare for possible maintenance."
    else:
        recommendation = "Continue routine monitoring."

    report = f"""
Advanced Predictive Maintenance Assistant
=========================================

Engine ID: {engine_id}
Current Cycle: {current_cycle}
Final Model Used: Ensemble of {best_tree_model_name} and LSTM

Predicted Remaining Useful Life: {predicted_rul:.1f} cycles
Risk Category: {risk}

Recommendation:
{recommendation}

Explanation:
This prediction combines a tree-based model using engineered trend features with an LSTM sequence model using the previous {WINDOW_SIZE} cycles of sensor data. The goal is to capture both tabular degradation patterns and temporal sensor behavior.
"""

    return report

In [ ]:
print(maintenance_report_advanced(1))

In [ ]:
def advanced_maintenance_chatbot():
    print("Advanced Jet Engine Predictive Maintenance Assistant")
    print(f"Best tree model: {best_tree_model_name}")
    print("Final prediction model: Ensemble of best tree model + LSTM")
    print("Type an engine number to get a report.")
    print("Type 'quit' to stop.")

    while True:
        user_input = input("\nEnter engine number: ")

        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye.")
            break

        try:
            engine_id = int(user_input)
            print(maintenance_report_advanced(engine_id))
        except:
            print("Please enter a valid engine number.")

In [ ]:
advanced_maintenance_chatbot()